# Fine-Tuning Bert

In [1]:
import numpy as np
import pandas as pd
import os
import re
from tqdm import tqdm
import torch
import torch.nn as nn
from torchinfo import summary
from transformers import BertModel, BertTokenizer
from torch.utils.data import DataLoader, TensorDataset
model_name = 'bert-base-uncased'

2025-10-17 19:49:47.199022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760730587.435738      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760730587.499070      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Preprocessing Data

In [2]:
def preprocess_text(text):
    text = text.lower()
    clean = re.compile('<.*?>')
    text = re.sub(clean, '', text)
    text = re.sub(r'[^a-z0-9\s\']', '', text)
    return text

In [3]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df['review'] = df['review'].apply(preprocess_text)

## Tokenize Data

In [4]:
tokenizer = BertTokenizer.from_pretrained(model_name)

encoded_inputs = tokenizer(list(df['review']), 
                               padding='max_length', 
                               truncation=True, 
                               max_length=319, 
                               return_tensors="pt")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Data Loader

In [5]:
dataset = TensorDataset(encoded_inputs['input_ids'], encoded_inputs['attention_mask'], torch.tensor(df['sentiment'].values))
data_loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4)

In [6]:
t, m, l = next(iter(data_loader))
print(t.shape, m.shape, l.shape)
print(t.dtype, m.dtype, l.dtype)

torch.Size([64, 319]) torch.Size([64, 319]) torch.Size([64])
torch.int64 torch.int64 torch.int64


## Bert transformer classifer

In [7]:
class Bert_model(nn.Module):
    def __init__(self, model_name, output_size, dropout=0.5):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Linear(self.bert.config.hidden_size, output_size)
    def forward(self, ids, mask):
        output = self.bert(input_ids=ids, attention_mask=mask)
        output_cls = output.last_hidden_state[:, 0, :]
        output_cls = self.dropout(output_cls)
        out = self.net(output_cls)
        out = torch.sigmoid(out)
        return out

bert_model = Bert_model(model_name, output_size=1)
summary(bert_model, input_size=[(64, 319), (64, 319)], dtypes=[torch.int64, torch.int64])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Layer (type:depth-idx)                                       Output Shape              Param #
Bert_model                                                   [64, 1]                   --
├─BertModel: 1-1                                             [64, 768]                 --
│    └─BertEmbeddings: 2-1                                   [64, 319, 768]            --
│    │    └─Embedding: 3-1                                   [64, 319, 768]            23,440,896
│    │    └─Embedding: 3-2                                   [64, 319, 768]            1,536
│    │    └─Embedding: 3-3                                   [1, 319, 768]             393,216
│    │    └─LayerNorm: 3-4                                   [64, 319, 768]            1,536
│    │    └─Dropout: 3-5                                     [64, 319, 768]            --
│    └─BertEncoder: 2-2                                      [64, 319, 768]            --
│    │    └─ModuleList: 3-6                                  --             

## Fine-Tune Model

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'working on {device}.')
def prepare_model(model):
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)  # Wrap model
    else:
        print("Using a single GPU or CPU.")
    return model.to(device)

working on cuda.


In [9]:
def save_model(model, path):
    if isinstance(model, nn.DataParallel):
        torch.save(model.module.state_dict(), path)
    else:
        torch.save(model.state_dict(), path)
    print(f'Model saved in {path}.')

In [10]:
bert_model = prepare_model(bert_model)

Using 2 GPUs!


In [11]:
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=5e-5)
criterion = nn.BCELoss()
epochs = 20

bert_model.train()
for epoch in range(epochs):
    train_loss = 0
    train_acc = 0
    total_train = 0
    for batch in tqdm(data_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        pred = bert_model(input_ids, attention_mask)
        loss = criterion(pred, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        pred_labels = (pred >= 0.5).int()
        train_acc += (pred_labels == labels).sum().item()
        total_train += labels.size()[0]
    train_acc /= total_train
    print(f'Epoch [{epoch+1}/{epochs}] loss {train_loss} acc {train_acc}.')

Epoch 1: 100%|██████████| 782/782 [25:33<00:00,  1.96s/it]


Epoch [1/20] loss 184.37416503578424 acc 0.90338.


Epoch 2: 100%|██████████| 782/782 [25:42<00:00,  1.97s/it]


Epoch [2/20] loss 96.57834025565535 acc 0.95524.


Epoch 3: 100%|██████████| 782/782 [25:48<00:00,  1.98s/it]


...

Epoch [18/20] loss 7.366009974874032 acc 0.99704.


Epoch 19: 100%|██████████| 782/782 [25:43<00:00,  1.97s/it]


Epoch [19/20] loss 7.030518213476171 acc 0.99706.


Epoch 20: 100%|██████████| 782/782 [25:42<00:00,  1.97s/it]

Epoch [20/20] loss 7.086765223284601 acc 0.99704.


In [12]:
save_model(bert_model, 'bert_sentiment_classifer.pth')

Model saved in bert_sentiment_classifer.pth.


## Test Model

In [13]:
def test_bert_model(model, text):
    test_model = model
    test_model.eval()
    encoded_input = tokenizer(
        text,
        return_tensors='pt',  # Return PyTorch tensors
        padding=True,
        truncation=True,
        max_length=128
    )
    with torch.no_grad():
        output = test_model(encoded_input['input_ids'],encoded_input['attention_mask'])
        pred_labels = (output >= 0.5).int()
    predicted_class_id = pred_labels.item()
    print(f"Text : {text}\n Predicted sentiment : {predicted_class_id}.")

test_bert_model(bert_model, "this product isn't bad.")
test_bert_model(bert_model, "this product is not that bad.")
test_bert_model(bert_model, "this product is good.")
test_bert_model(bert_model, "this product is not good.")

Text : this product isn't bad.
 Predicted sentiment : 1.
Text : this product is not that bad.
 Predicted sentiment : 0.
Text : this product is good.
 Predicted sentiment : 1.
Text : this product is not good.
 Predicted sentiment : 0.


In [14]:
def test_example_texts(model):
    examples = [
        "I love this product!",
        "This service was absolutely awful.",
        "The results were simply excellent.",
        "What a disappointment.",
        "The performance wasn't great.",
        "I would not recommend this item.",
        "It's not the worst thing I've ever bought.",
        "The food was barely edible.",
        "Sure, the battery life is great... if you want to recharge it every hour.",
        "I'm so thrilled about this broken app.",
        "The movie was long, but not boring.",
        "It works, but the quality is terrible."
    ]
    for ex in examples:
        text = preprocess_text(ex)
        test_bert_model(model, text)
        

In [15]:
test_example_texts(bert_model)

Text : i love this product
 Predicted sentiment : 1.
Text : this service was absolutely awful
 Predicted sentiment : 0.
Text : the results were simply excellent
 Predicted sentiment : 1.
Text : what a disappointment
 Predicted sentiment : 0.
Text : the performance wasn't great
 Predicted sentiment : 0.
Text : i would not recommend this item
 Predicted sentiment : 0.
Text : it's not the worst thing i've ever bought
 Predicted sentiment : 0.
Text : the food was barely edible
 Predicted sentiment : 0.
Text : sure the battery life is great if you want to recharge it every hour
 Predicted sentiment : 1.
Text : i'm so thrilled about this broken app
 Predicted sentiment : 1.
Text : the movie was long but not boring
 Predicted sentiment : 1.
Text : it works but the quality is terrible
 Predicted sentiment : 1.


# Reference Model

In [16]:
%%writefile bert_sentiment_classifier.py

import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
model_name = 'bert-base-uncased'
import re

def preprocess_text(text):
    text = text.lower()
    clean = re.compile('<.*?>')
    text = re.sub(clean, '', text)
    text = re.sub(r'[^a-z0-9\s\']', '', text)
    return text
    
class Bert_model(nn.Module):
    def __init__(self, output_size=1, dropout=0.5):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Linear(self.bert.config.hidden_size, output_size)

        self.tokenizer = BertTokenizer.from_pretrained(model_name)
    def prepare_input(self, text):
        text = preprocess_text(text)
        encoded_input = self.tokenizer(
            text,
            return_tensors='pt',  # Return PyTorch tensors
            padding=True,
            truncation=True,
            max_length=128
        )
        return encoded_input['input_ids'], encoded_input['attention_mask']
    def load_model(self, model_path):
        state_dict = torch.load(model_path)
        self.load_state_dict(state_dict)
    def forward(self, ids, mask):
        output = self.bert(input_ids=ids, attention_mask=mask)
        output_cls = output.last_hidden_state[:, 0, :]
        output_cls = self.dropout(output_cls)
        out = self.net(output_cls)
        out = torch.sigmoid(out)
        return out

    def predict_sentiment(self, text):
        ids, masks = self.prepare_input(text)
        with torch.no_grad():
            output = self.forward(ids,masks)
            pred_labels = (output >= 0.5).int().item()
        print(f"Text : {text}\n Predicted sentiment : {pred_labels}.")

Writing bert_sentiment_classifier.py


In [ ]:
import importlib.util
def get_module_from_file(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    mask_detector_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mask_detector_module)
    return mask_detector_module

In [18]:
bert_classifier_module = get_module_from_file('bert_classifier_module', '/kaggle/working/bert_sentiment_classifier.py')
loaded_bert = bert_classifier_module.Bert_model()
loaded_bert.load_model('/kaggle/working/bert_sentiment_classifer.pth')

In [19]:
loaded_bert.predict_sentiment("this is very good.")
loaded_bert.predict_sentiment("this is very bad.")

Text : this is very good.
 Predicted sentiment : 1.
Text : this is very bad.
 Predicted sentiment : 0.
